# PCA and UMAP for Expanded Embedding Tables

This notebook is for downstream exploratory analysis after embedding vectors have already been expanded into CSV columns such as `emb_0000 ... emb_1535`.

Intended use:
- load a full metadata table or an embeddings-only table
- detect embedding columns automatically from a prefix
- optionally standardize embeddings
- run PCA first
- run UMAP to 2D from the PCA representation
- visualize embeddings colored by one or more metadata columns
- optionally explore KMeans clusters

This notebook was written from the expected CSV schema and was not tested on your real HPC outputs.

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

try:
    import umap
except ImportError as exc:
    raise ImportError(
        "This notebook requires umap-learn. Install it in the environment before running the UMAP cells."
    ) from exc

plt.rcParams["figure.dpi"] = 120


## User Configuration

In [ ]:
REPO_ROOT = None  # Optional: set this if you open the notebook from outside the repo root.
INPUT_CSV = Path("outputs/conic_liz/embedding_tables/embed_morph_with_vectors.csv")
EMBEDDING_PREFIX = "emb_"

PCA_N_COMPONENTS = 50
STANDARDIZE_EMBEDDINGS = True

UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
RANDOM_SEED = 42

FIGSIZE = (8, 6)
POINT_SIZE = 12
ALPHA = 0.7

COLOR_COLUMN = "dominant_class"
BATCH_COLOR_COLUMNS = [
    "dominant_class",
    "richness_label",
    "count_total",
    "total_nuclei",
    "foreground_fraction",
    "mean_area",
    "median_area",
    "std_area",
    "mean_eccentricity",
    "mean_solidity",
    "mean_circularity",
    "fraction_neutrophil",
    "fraction_epithelial",
    "fraction_lymphocyte",
    "fraction_plasma",
    "fraction_eosinophil",
    "fraction_connective",
    "dominant_fraction",
]

RUN_KMEANS = False
KMEANS_N_CLUSTERS = 8
KMEANS_LABEL_COLUMN = "kmeans_cluster"

PLOT_OUTPUT_DIR = Path("outputs/conic_liz/embedding_tables/plots")


## Load Data

In [ ]:
def find_repo_root(start: Path) -> Path | None:
    start_path = start.expanduser().resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / ".git").exists():
            return candidate
    return None


def resolve_project_path(path_like: Path, repo_root: Path | None = None) -> Path:
    path = Path(path_like).expanduser()
    if path.is_absolute():
        return path.resolve()

    candidates = []
    if repo_root is not None:
        candidates.append((repo_root / path).resolve())
    candidates.append(path.resolve())

    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


repo_root = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT is not None else find_repo_root(Path.cwd())
input_csv = resolve_project_path(INPUT_CSV, repo_root=repo_root)
plot_output_dir = resolve_project_path(PLOT_OUTPUT_DIR, repo_root=repo_root)

if not input_csv.exists():
    raise FileNotFoundError(
        f"Input CSV does not exist: {input_csv}. Update INPUT_CSV in the config cell before running further."
    )

if repo_root is not None:
    print(f"Resolved repo root: {repo_root}")
print(f"Resolved input CSV: {input_csv}")
print(f"Plot output directory for optional saves: {plot_output_dir}")

df = pd.read_csv(input_csv)
print(f"Loaded {len(df):,} rows x {len(df.columns):,} columns from {input_csv}")
df.head()


## Detect Embedding Columns

In [ ]:
def embedding_sort_key(column_name: str):
    suffix = str(column_name)[len(EMBEDDING_PREFIX):]
    return (0, int(suffix)) if suffix.isdigit() else (1, str(column_name))


embedding_cols = [column for column in df.columns if str(column).startswith(EMBEDDING_PREFIX)]
if not embedding_cols:
    raise ValueError(
        f"No embedding columns were found with prefix {EMBEDDING_PREFIX!r}. "
        "Update EMBEDDING_PREFIX or check the input CSV."
    )

embedding_cols = sorted(embedding_cols, key=embedding_sort_key)
metadata_cols = [column for column in df.columns if column not in embedding_cols]

print(f"Detected {len(embedding_cols):,} embedding columns.")
print(f"First embedding columns: {embedding_cols[:5]}")
print(f"Detected {len(metadata_cols):,} metadata columns.")

missing_batch_cols = [column for column in BATCH_COLOR_COLUMNS if column not in df.columns]
available_batch_cols = [column for column in BATCH_COLOR_COLUMNS if column in df.columns]
print(f"Available configured batch metadata columns: {available_batch_cols}")
if missing_batch_cols:
    print(f"Configured batch metadata columns not present in this CSV: {missing_batch_cols}")

if COLOR_COLUMN not in df.columns:
    print(
        f"COLOR_COLUMN={COLOR_COLUMN!r} is not present in this CSV. "
        "This is expected if you loaded the sample_id-only table."
    )


## Build Embedding Matrix

In [ ]:
X = df[embedding_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
if X.ndim != 2 or X.shape[1] == 0:
    raise ValueError(f"Expected a 2D embedding matrix with at least one column, got shape {X.shape}.")

non_finite_mask = ~np.isfinite(X)
if non_finite_mask.any():
    raise ValueError(
        f"Embedding matrix contains {non_finite_mask.sum():,} non-finite values. "
        "Check the exported embedding CSV before running PCA/UMAP."
    )

if X.shape[0] < 2:
    raise ValueError("At least two rows are required for PCA/UMAP exploration.")

print(f"Embedding matrix shape: {X.shape}")


## Optional Standardization

In [ ]:
if STANDARDIZE_EMBEDDINGS:
    scaler = StandardScaler()
    X_model = scaler.fit_transform(X)
    print("Applied StandardScaler to embedding columns before PCA.")
else:
    scaler = None
    X_model = X.copy()
    print("Using raw embeddings without standardization.")


## PCA

In [ ]:
max_pca_components = min(PCA_N_COMPONENTS, X_model.shape[0], X_model.shape[1])
if max_pca_components < 2:
    raise ValueError(
        f"Need at least 2 PCA components for plotting, but only {max_pca_components} component(s) are possible."
    )
if max_pca_components != PCA_N_COMPONENTS:
    print(f"Adjusted PCA components from {PCA_N_COMPONENTS} to {max_pca_components} based on data shape.")

pca = PCA(n_components=max_pca_components, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_model)
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_explained_variance = np.cumsum(explained_variance_ratio)

print(f"PCA output shape: {X_pca.shape}")
print(f"First five explained variance ratios: {explained_variance_ratio[:5]}")
print(f"Cumulative explained variance at component {len(cumulative_explained_variance)}: {cumulative_explained_variance[-1]:.4f}")

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(np.arange(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, marker="o", linewidth=1.5)
ax.set_xlabel("PCA component")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("PCA cumulative explained variance")
ax.grid(alpha=0.3)
plt.tight_layout()
# plt.savefig(plot_output_dir / "pca_cumulative_explained_variance.png", bbox_inches="tight")
plt.show()


## UMAP

In [ ]:
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric="euclidean",
    random_state=RANDOM_SEED,
)
X_umap = umap_model.fit_transform(X_pca)
print(f"UMAP output shape: {X_umap.shape}")


## Build Plotting DataFrame

In [ ]:
plot_df = df.copy()
plot_df["pca_1"] = X_pca[:, 0]
plot_df["pca_2"] = X_pca[:, 1]
plot_df["umap_1"] = X_umap[:, 0]
plot_df["umap_2"] = X_umap[:, 1]

plot_df[["pca_1", "pca_2", "umap_1", "umap_2"]].head()


## Plotting Helpers

In [ ]:
def ensure_columns_present(frame: pd.DataFrame, columns, *, context: str = ""):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        prefix = f"{context}: " if context else ""
        raise KeyError(f"{prefix}missing columns: {missing}")


def plot_embedding_scatter(
    frame: pd.DataFrame,
    x_col: str,
    y_col: str,
    color_col: str | None = None,
    *,
    ax=None,
    title: str | None = None,
    figsize=FIGSIZE,
    point_size: float = POINT_SIZE,
    alpha: float = ALPHA,
    max_legend_items: int = 25,
):
    ensure_columns_present(frame, [x_col, y_col], context="plot_embedding_scatter")
    subset = frame.loc[frame[x_col].notna() & frame[y_col].notna()].copy()
    if subset.empty:
        raise ValueError(f"No rows with non-null coordinates for {x_col!r} and {y_col!r}.")

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    if color_col is None:
        ax.scatter(subset[x_col], subset[y_col], s=point_size, alpha=alpha, c="steelblue", linewidths=0)
    else:
        ensure_columns_present(subset, [color_col], context="plot_embedding_scatter")
        color_series = subset[color_col]

        if pd.api.types.is_numeric_dtype(color_series):
            numeric_values = pd.to_numeric(color_series, errors="coerce")
            valid_color_mask = numeric_values.notna()

            if (~valid_color_mask).any():
                ax.scatter(
                    subset.loc[~valid_color_mask, x_col],
                    subset.loc[~valid_color_mask, y_col],
                    s=point_size,
                    alpha=alpha,
                    c="lightgray",
                    linewidths=0,
                    label="Missing",
                )

            scatter = ax.scatter(
                subset.loc[valid_color_mask, x_col],
                subset.loc[valid_color_mask, y_col],
                c=numeric_values.loc[valid_color_mask],
                cmap="viridis",
                s=point_size,
                alpha=alpha,
                linewidths=0,
            )
            cbar = fig.colorbar(scatter, ax=ax)
            cbar.set_label(color_col)
        else:
            categorical_values = color_series.astype("object").where(color_series.notna(), "Missing").astype(str)
            if categorical_values.nunique() > max_legend_items:
                top_values = categorical_values.value_counts().nlargest(max_legend_items - 1).index.tolist()
                categorical_values = categorical_values.where(categorical_values.isin(top_values), "Other")

            categories = list(pd.Index(categorical_values.unique()))
            cmap_name = "tab20" if len(categories) <= 20 else "gist_ncar"
            cmap = plt.get_cmap(cmap_name, len(categories))
            color_lookup = {category: cmap(index) for index, category in enumerate(categories)}
            point_colors = [color_lookup[value] for value in categorical_values]

            ax.scatter(subset[x_col], subset[y_col], s=point_size, alpha=alpha, c=point_colors, linewidths=0)
            handles = [
                Line2D(
                    [0],
                    [0],
                    marker="o",
                    color="w",
                    markerfacecolor=color_lookup[category],
                    label=str(category),
                    markersize=6,
                )
                for category in categories
            ]
            ax.legend(handles=handles, title=color_col, bbox_to_anchor=(1.02, 1), loc="upper left")

    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title or f"{y_col} vs {x_col}")
    ax.grid(alpha=0.25)
    return ax


def plot_pca_and_umap(frame: pd.DataFrame, color_col: str | None = None):
    color_label = color_col if color_col is not None else "no color overlay"
    fig, axes = plt.subplots(1, 2, figsize=(FIGSIZE[0] * 2, FIGSIZE[1]))
    plot_embedding_scatter(frame, "pca_1", "pca_2", color_col=color_col, ax=axes[0], title=f"PCA colored by {color_label}")
    plot_embedding_scatter(frame, "umap_1", "umap_2", color_col=color_col, ax=axes[1], title=f"UMAP colored by {color_label}")
    plt.tight_layout()
    return fig, axes


## Single-Feature Coloring Examples

In [ ]:
if COLOR_COLUMN is None:
    fig, axes = plot_pca_and_umap(plot_df, color_col=None)
    # plt.savefig(plot_output_dir / "uncolored_pca_umap.png", bbox_inches="tight")
    plt.show()
elif COLOR_COLUMN in plot_df.columns:
    fig, axes = plot_pca_and_umap(plot_df, color_col=COLOR_COLUMN)
    # plt.savefig(plot_output_dir / f"{COLOR_COLUMN}_pca_umap.png", bbox_inches="tight")
    plt.show()
else:
    print(
        f"COLOR_COLUMN={COLOR_COLUMN!r} is not present in this dataframe. "
        "Choose another column in the config cell if you want a single-feature plot."
    )


## Multi-Feature Loop Examples

In [ ]:
available_batch_cols = [column for column in BATCH_COLOR_COLUMNS if column in plot_df.columns]
missing_batch_cols = [column for column in BATCH_COLOR_COLUMNS if column not in plot_df.columns]

print(f"Batch columns available for plotting: {available_batch_cols}")
if missing_batch_cols:
    print(f"Batch columns missing from this dataframe: {missing_batch_cols}")

if not available_batch_cols:
    print("No configured batch metadata columns are present in this dataframe.")
else:
    for color_col in available_batch_cols:
        fig, axes = plot_pca_and_umap(plot_df, color_col=color_col)
        # plt.savefig(plot_output_dir / f"{color_col}_pca_umap.png", bbox_inches="tight")
        plt.show()


## Optional KMeans Exploratory Section

In [ ]:
if RUN_KMEANS:
    if KMEANS_N_CLUSTERS < 2:
        raise ValueError(f"KMEANS_N_CLUSTERS must be at least 2, got {KMEANS_N_CLUSTERS}.")

    kmeans = KMeans(n_clusters=KMEANS_N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
    plot_df[KMEANS_LABEL_COLUMN] = kmeans.fit_predict(X_pca)
    print(f"Added cluster labels to column {KMEANS_LABEL_COLUMN!r}.")

    fig, ax = plt.subplots(figsize=FIGSIZE)
    plot_embedding_scatter(
        plot_df,
        "umap_1",
        "umap_2",
        color_col=KMEANS_LABEL_COLUMN,
        ax=ax,
        title=f"UMAP colored by {KMEANS_LABEL_COLUMN}",
    )
    plt.tight_layout()
    # plt.savefig(plot_output_dir / f"{KMEANS_LABEL_COLUMN}_umap.png", bbox_inches="tight")
    plt.show()
else:
    print("Set RUN_KMEANS = True in the config cell to run the optional clustering section.")


## Interpretation Notes

- PCA is useful for checking how much variance is captured before nonlinear visualization.
- UMAP often highlights local neighborhoods more strongly than PCA, so separated islands do not automatically imply discrete biology.
- Numeric color overlays help check whether morphology or composition variables vary smoothly across the embedding manifold.
- Categorical overlays such as `dominant_class` or `richness_label` help assess whether known labels align with embedding structure.
- If you load the embeddings-only CSV, metadata-driven coloring will be limited to whatever columns remain in that file.
- This notebook assumes embedding columns are already flattened in the CSV and does not validate the upstream HPC export beyond basic column and numeric checks.